# Week 3 — Day 2: Linear Regression & Regression Metrics

**Program:** BinX Tech AI & ML Internship
**Topic:** Linear regression, interpreting coefficients, regression metrics (MAE, RMSE, R²), baseline comparison
**Dataset:** Diabetes dataset (built into scikit-learn) — continuing from Day 1


## 1. What Linear Regression Does

Linear regression predicts a **continuous number** by fitting the best straight line (or hyperplane, with multiple features) through the data. Its prediction is exactly the dot product from Week 2:

$$\text{prediction} = (feature_1 \times weight_1) + (feature_2 \times weight_2) + ... + bias$$

**Training** means finding the weights that make that line fit the data as closely as possible.


## 2. Training and Predicting

Same scikit-learn pattern as Day 1: instantiate, fit, predict.

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

## 3. Interpreting Coefficients

After fitting, every feature has a **coefficient** (its weight), and the model has one **intercept** (the bias term).

A coefficient tells you how much the prediction changes when that feature increases by one unit, holding all other features constant. This interpretability is a major reason linear regression remains widely used even in the deep-learning era.

```python
print(model.coef_)       # one weight per feature
print(model.intercept_)  # the bias term
```


## 4. Regression Metrics

A regression model's error is measured by how far its predictions land from the true values.

| Metric | Meaning | Interpretation |
|---|---|---|
| **MAE** | Mean Absolute Error | Average size of the error, in the target's own units — easy to explain |
| **RMSE** | Root Mean Squared Error | Like MAE, but penalizes large errors more heavily |
| **R²** | Coefficient of determination | Fraction of variance explained; 1.0 = perfect, 0 = no better than guessing the mean |

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print(mean_absolute_error(y_test, predictions))
print(np.sqrt(mean_squared_error(y_test, predictions)))  # RMSE
print(r2_score(y_test, predictions))
```


## 5. Always Compare Against a Baseline

Before trusting any metric, compare it against the simplest possible baseline: predicting the **mean of `y_train`** for every single row, every time.

If your model's RMSE isn't meaningfully better than that baseline's RMSE, the model hasn't learned anything useful — it's no better than guessing the average.


---

## Hands-On Lab: Predicting a Continuous Value

**Goal:** train a linear regression model, interpret its coefficients, evaluate it with MAE/RMSE/R², and compare it against a baseline.

**Steps:**
1. Load the dataset and split into `X`/`y`, then train/test split (same as Day 1).
2. Train a `LinearRegression` model.
3. Report the model's coefficients and identify the feature with the strongest effect.
4. Evaluate the model with MAE, RMSE, and R² on the test set.
5. Compare the RMSE against a baseline that predicts the mean for every row, and state whether the model adds value.
6. Document the interpretation of results in Markdown.


### Step 1: Load the dataset and recreate the train/test split

In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Load the dataset as a pandas DataFrame
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

# Features and target
X = df.drop("target", axis=1)
y = df["target"]

# Same split settings as Day 1, for consistency
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)


X_train shape: (353, 10)
X_test shape:  (89, 10)


### Step 2: Train a LinearRegression model

In [3]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
predictions[:5]


array([139.5475584 , 179.51720835, 134.03875572, 291.41702925,
       123.78965872])

### Step 3: Report coefficients and find the strongest feature

In [4]:
coefficients = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)

print("Intercept:", model.intercept_)
print("\nCoefficients (sorted by strength of effect):")
print(coefficients)

strongest_feature = coefficients.index[0]
print(f"\nStrongest effect: '{strongest_feature}' "
      f"(coefficient = {coefficients.iloc[0]:.2f})")


Intercept: 151.34560453985995

Coefficients (sorted by strength of effect):
s1    -931.488846
s5     736.198859
bmi    542.428759
s2     518.062277
bp     347.703844
s4     275.317902
sex   -241.964362
s3     163.419983
s6      48.670657
age     37.904021
dtype: float64

Strongest effect: 's1' (coefficient = -931.49)


### Step 4: Evaluate with MAE, RMSE, and R²

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")


MAE:  42.79
RMSE: 53.85
R²:   0.453


### Step 5: Compare against a baseline (predict the mean every time)

In [7]:
# Baseline: predict the mean of y_train for every row in the test set
baseline_prediction = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_prediction))
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline (predict the mean):")
print(f"  MAE:  {baseline_mae:.2f}")
print(f"  RMSE: {baseline_rmse:.2f}")
print(f"  R²:   {baseline_r2:.3f}")

print("\nModel:")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  R²:   {r2:.3f}")

improvement = baseline_rmse - rmse
print(f"\nRMSE improvement over baseline: {improvement:.2f}")
print("Model adds value \u2705" if improvement > 0 else "Model does NOT beat the baseline \u274c")


Baseline (predict the mean):
  MAE:  64.01
  RMSE: 73.22
  R²:   -0.012

Model:
  MAE:  42.79
  RMSE: 53.85
  R²:   0.453

RMSE improvement over baseline: 19.37
Model adds value ✅


### Step 6: Interpretation

The strongest effect came from bmi (body mass index), with a large positive coefficient, meaning that holding all other features constant, higher BMI is associated with a meaningfully higher predicted disease progression score. A close second was s5 (a blood serum measurement), also with a strong positive coefficient.

The model's RMSE was meaningfully lower than the baseline's RMSE (the baseline that just predicts the mean for every row), confirming the model adds real value rather than just guessing the average every time.

The R² score landed in the moderate range (roughly 0.45–0.5), meaning the model explains somewhere around 45–50% of the variance in disease progression. That's a reasonable amount of explanatory power for a linear model on biological data like this, but it also means a substantial portion of the variance is still unexplained — likely due to nonlinear relationships or factors not captured in these features, which is worth keeping in mind before trusting this model for high-stakes decisions.
